# 02 Residual Stream / Logit Lens / Activation Patching

## このノートブックの目的

Qwen3-4B の内部計算を 3 つの視点から観察する。

| 概念 | 概要 |
|------|------|
| **Residual Stream** | 各 Transformer layer の入出力となる hidden state のベクトル系列 |
| **Logit Lens** | 各 layer 後の hidden state を lm_head に通して「その時点でのモデルの予測」を読む |
| **Activation Patching** | clean run の hidden state を corrupt run に注入し、各 layer の因果的寄与を測る |

### 実験設定

| | プロンプト | 期待する次トークン |
|---|---|---|
| **clean** | `"The capital of Japan is"` | ` Tokyo` |
| **corrupt** | `"The capital of France is"` | ` Paris` |

clean run の hidden state を layer k で corrupt run に注入したとき、出力がどう変化するかを計測する。


## 1. 環境セットアップ

In [ ]:
%matplotlib inline
import sys, json
from pathlib import Path
import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# プロジェクトルートを sys.path に追加
# sandbox/ から実行した場合は1つ上がる
_cwd = Path(".").resolve()
project_root = _cwd.parent if _cwd.name == "sandbox" else _cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts.common import load_config, resolve_scratch_dir, ensure_dir

cfg = load_config()
model_id = cfg["model_id"]
print(f"model_id : {model_id}")

# デバイス選択
if torch.cuda.is_available():
    device = "cuda"
    torch_dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    torch_dtype = torch.float16
else:
    device = "cpu"
    torch_dtype = torch.float32
print(f"device   : {device}")
print(f"dtype    : {torch_dtype}")

outputs_dir = project_root / "outputs"
ensure_dir(outputs_dir)
print(f"outputs  : {outputs_dir}")


### ヘルパー関数

In [ ]:
def repr_piece(piece: str) -> str:
    """Leading space などが見えるよう repr 表示"""
    return repr(piece)


def topk_table(logits: torch.Tensor, tokenizer, k: int = 10) -> pd.DataFrame:
    """logits [vocab] -> top-k DataFrame"""
    probs = torch.softmax(logits.float(), dim=-1)
    top_vals, top_ids = torch.topk(probs, k)
    rows = []
    for rank, (tid, prob) in enumerate(zip(top_ids.tolist(), top_vals.tolist()), start=1):
        piece = tokenizer.decode([tid])
        rows.append({
            "rank": rank,
            "token_id": tid,
            "piece": repr_piece(piece),
            "logit": logits[tid].item(),
            "prob": prob,
        })
    return pd.DataFrame(rows)


def show_token_table(text: str, tokenizer) -> pd.DataFrame:
    """テキストをトークナイズして位置・ID・piece の表を返す"""
    ids = tokenizer.encode(text, add_special_tokens=False)
    rows = []
    for pos, tid in enumerate(ids):
        piece = tokenizer.convert_ids_to_tokens(tid)
        decoded = tokenizer.decode([tid])
        rows.append({
            "pos": pos,
            "token_id": tid,
            "piece": repr_piece(piece),
            "decoded": repr_piece(decoded),
        })
    return pd.DataFrame(rows)


def logit_lens(hidden_states, k: int, pos: int, model) -> torch.Tensor:
    """
    hidden_states[k] の position pos を lm_head に通して logits [vocab] を返す。

    k < K : model.model.norm を適用してから lm_head に通す
    k = K : hs[K] は norm 後なので直接 lm_head に通す
    """
    K = len(hidden_states) - 1
    hs = hidden_states[k]   # [1, seq, hidden]
    with torch.no_grad():
        if k < K:
            normed = model.model.norm(hs[:, pos:pos+1, :])  # [1, 1, hidden]
            logits = model.lm_head(normed)[:, 0, :]          # [1, vocab]
        else:
            normed = hs[:, pos, :]        # [1, hidden]
            logits = model.lm_head(normed)  # [1, vocab]
    return logits[0]  # [vocab]


def metric(logits: torch.Tensor, clean_id: int, corrupt_id: int) -> float:
    """logit(clean_answer) - logit(corrupt_answer) を返す"""
    return (logits[clean_id] - logits[corrupt_id]).item()


## 2. モデルとトークナイザーの読み込み

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedModel

print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
print(f"  vocab_size = {tokenizer.vocab_size}")

print("Loading model ...")
model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    attn_implementation="eager",
)
model.to(device)  # pyright: ignore[reportArgumentType]
model.eval()
print(f"  device     : {next(model.parameters()).device}")
print(f"  dtype      : {next(model.parameters()).dtype}")

K = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
print(f"  layers K   : {K}")
print(f"  hidden_size: {hidden_size}")


## 3. トークンテーブル

In [ ]:
CLEAN_PROMPT   = "The capital of Japan is"
CORRUPT_PROMPT = "The capital of France is"
CLEAN_ANSWER   = " Tokyo"
CORRUPT_ANSWER = " Paris"

# answer が単一トークンかどうか確認
clean_ans_ids   = tokenizer.encode(CLEAN_ANSWER,   add_special_tokens=False)
corrupt_ans_ids = tokenizer.encode(CORRUPT_ANSWER, add_special_tokens=False)
print(f"CLEAN_ANSWER   {CLEAN_ANSWER!r:12s} -> ids = {clean_ans_ids}")
print(f"CORRUPT_ANSWER {CORRUPT_ANSWER!r:12s} -> ids = {corrupt_ans_ids}")
if len(clean_ans_ids) != 1 or len(corrupt_ans_ids) != 1:
    print("[warning] answer string is not a single token")

CLEAN_ANS_ID   = clean_ans_ids[0]
CORRUPT_ANS_ID = corrupt_ans_ids[0]
print(f"\nCLEAN_ANS_ID   = {CLEAN_ANS_ID}  ({repr_piece(tokenizer.decode([CLEAN_ANS_ID]))})")
print(f"CORRUPT_ANS_ID = {CORRUPT_ANS_ID}  ({repr_piece(tokenizer.decode([CORRUPT_ANS_ID]))})")


In [ ]:
print("=== clean prompt token table ===")
df_clean_tok = show_token_table(CLEAN_PROMPT, tokenizer)
display(df_clean_tok)
clean_pos = len(df_clean_tok) - 1
print(f"\nclean_pos (last token position) = {clean_pos}")

print("\n=== corrupt prompt token table ===")
df_corrupt_tok = show_token_table(CORRUPT_PROMPT, tokenizer)
display(df_corrupt_tok)
corrupt_pos = len(df_corrupt_tok) - 1
print(f"\ncorrupt_pos (last token position) = {corrupt_pos}")


## 4. Forward pass (baseline)

In [ ]:
# clean run
clean_inputs = tokenizer(CLEAN_PROMPT, return_tensors="pt").to(device)
with torch.no_grad():
    clean_outputs = model(
        **clean_inputs,
        output_hidden_states=True,
        output_attentions=False,
        use_cache=False,
    )

clean_hs     = clean_outputs.hidden_states           # tuple, len = K+1
clean_logits = clean_outputs.logits[0, clean_pos, :].float()
clean_probs  = torch.softmax(clean_logits, dim=-1)
print(f"clean run:")
print(f"  top-1   = {repr_piece(tokenizer.decode([clean_logits.argmax().item()]))}")
print(f"  P(Tokyo) = {clean_probs[CLEAN_ANS_ID]:.4f}")
print(f"  P(Paris) = {clean_probs[CORRUPT_ANS_ID]:.4f}")


In [ ]:
# corrupt run
corrupt_inputs = tokenizer(CORRUPT_PROMPT, return_tensors="pt").to(device)
with torch.no_grad():
    corrupt_outputs = model(
        **corrupt_inputs,
        output_hidden_states=True,
        output_attentions=False,
        use_cache=False,
    )

corrupt_hs     = corrupt_outputs.hidden_states
corrupt_logits = corrupt_outputs.logits[0, corrupt_pos, :].float()
corrupt_probs  = torch.softmax(corrupt_logits, dim=-1)
print(f"corrupt run:")
print(f"  top-1   = {repr_piece(tokenizer.decode([corrupt_logits.argmax().item()]))}")
print(f"  P(Tokyo) = {corrupt_probs[CLEAN_ANS_ID]:.4f}")
print(f"  P(Paris) = {corrupt_probs[CORRUPT_ANS_ID]:.4f}")


In [ ]:
print("=== clean top-10 ===")
display(topk_table(clean_logits, tokenizer, k=10))
print("\n=== corrupt top-10 ===")
display(topk_table(corrupt_logits, tokenizer, k=10))


## 5. Hidden states の形状

In [ ]:
print(f"len(clean_hs) = {len(clean_hs)}  (= K+1 = {K}+1)")
print(f"hs[0].shape   = {tuple(clean_hs[0].shape)}  <- embed_tokens 出力")
print(f"hs[1].shape   = {tuple(clean_hs[1].shape)}  <- layer 0 出力")
print(f"hs[K].shape   = {tuple(clean_hs[K].shape)}  <- norm 後 (k=K)")
print()
print("Residual stream のインデックス対応:")
print("  hs[0]   = embed_tokens(input_ids)")
print("  hs[k]   = layers[k-1] の出力  (1 <= k <= K)")
print(f"  hs[{K}]  = model.model.norm(hs[{K}-1])  ← lm_head の直前")
print()
for k in [0, 1, 2, K - 1, K]:
    if k == 0:
        site = "embed_tokens"
    elif k < K:
        site = f"layer {k-1} 出力"
    else:
        site = f"norm 後 (k=K={K})"
    print(f"  hs[{k:2d}]: {site}")


## 6. lm_head の動作確認

In [ ]:
# hs[K] (norm 後) を lm_head に通した結果と model output logits を比較
ll_logits_K = logit_lens(clean_hs, K, clean_pos, model)
diff = (ll_logits_K - clean_logits).abs().max().item()
print(f"logit_lens(k=K) vs model logits: max abs diff = {diff:.6f}")
print("  diff ≈ 0 なら logit_lens の実装が正しい")
print()
display(topk_table(ll_logits_K, tokenizer, k=5))


## 7. Logit Lens — clean run

In [ ]:
print(f"{'k':>4}  {'site':>7}  {'top-1 piece':20s}  {'P(Tokyo)':>9}  {'P(Paris)':>9}")
print("-" * 60)
ll_rows = []
for k in range(K + 1):
    ll_logits = logit_lens(clean_hs, k, clean_pos, model)
    ll_probs  = torch.softmax(ll_logits.float(), dim=-1)
    top1_id   = ll_logits.argmax().item()
    top1_piece = tokenizer.decode([top1_id])
    p_tokyo   = ll_probs[CLEAN_ANS_ID].item()
    p_paris   = ll_probs[CORRUPT_ANS_ID].item()
    site = "embed" if k == 0 else (f"L{k-1:02d}" if k < K else "norm")
    print(f"  k={k:2d}  {site:>7}  {repr_piece(top1_piece):20s}  {p_tokyo:9.4f}  {p_paris:9.4f}")
    ll_rows.append({
        "k": k, "site": site,
        "top1_piece": top1_piece,
        "p_tokyo": p_tokyo,
        "p_paris": p_paris,
    })

df_ll_clean = pd.DataFrame(ll_rows)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_ll_clean["k"], df_ll_clean["p_tokyo"], label="P( Tokyo)", marker="o", markersize=3)
ax.plot(df_ll_clean["k"], df_ll_clean["p_paris"], label="P( Paris)", marker="s", markersize=3, linestyle="--")
ax.set_xlabel("layer k  (0=embed, K=norm)")
ax.set_ylabel("probability")
ax.set_title("Logit Lens — clean run  (The capital of Japan is)")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(outputs_dir / "nb02_logit_lens_clean.png", dpi=150)
plt.show()
print("Saved: outputs/nb02_logit_lens_clean.png")


## 8. Logit Lens 比較 — clean vs corrupt

In [ ]:
ll_rows_corrupt = []
for k in range(K + 1):
    ll_logits = logit_lens(corrupt_hs, k, corrupt_pos, model)
    ll_probs  = torch.softmax(ll_logits.float(), dim=-1)
    top1_id   = ll_logits.argmax().item()
    top1_piece = tokenizer.decode([top1_id])
    p_tokyo   = ll_probs[CLEAN_ANS_ID].item()
    p_paris   = ll_probs[CORRUPT_ANS_ID].item()
    site = "embed" if k == 0 else (f"L{k-1:02d}" if k < K else "norm")
    ll_rows_corrupt.append({
        "k": k, "site": site,
        "top1_piece": top1_piece,
        "p_tokyo": p_tokyo,
        "p_paris": p_paris,
    })

df_ll_corrupt = pd.DataFrame(ll_rows_corrupt)
print("Done.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

ax = axes[0]
ax.plot(df_ll_clean["k"], df_ll_clean["p_tokyo"], label="P( Tokyo)", marker="o", markersize=3)
ax.plot(df_ll_clean["k"], df_ll_clean["p_paris"], label="P( Paris)", marker="s", markersize=3, linestyle="--")
ax.set_title("Logit Lens — clean  (Japan)")
ax.set_xlabel("layer k")
ax.set_ylabel("probability")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(df_ll_corrupt["k"], df_ll_corrupt["p_tokyo"], label="P( Tokyo)", marker="o", markersize=3)
ax.plot(df_ll_corrupt["k"], df_ll_corrupt["p_paris"], label="P( Paris)", marker="s", markersize=3, linestyle="--")
ax.set_title("Logit Lens — corrupt  (France)")
ax.set_xlabel("layer k")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(outputs_dir / "nb02_logit_lens_comparison.png", dpi=150)
plt.show()
print("Saved: outputs/nb02_logit_lens_comparison.png")


### 補足：logit-difference metric で見る logit lens

確率 plot は後半層で 0/1 に飽和するため、前半・中盤の変化が見えにくい場合がある。
`metric_k = logit_k(" Tokyo") - logit_k(" Paris")` を clean/corrupt で並べると、変化がより見やすくなる。


In [ ]:
# logit-difference metric plot (clean / corrupt)
ll_metric_clean   = [
    metric(logit_lens(clean_hs,   k, clean_pos,   model), CLEAN_ANS_ID, CORRUPT_ANS_ID)
    for k in range(K + 1)
]
ll_metric_corrupt = [
    metric(logit_lens(corrupt_hs, k, corrupt_pos, model), CLEAN_ANS_ID, CORRUPT_ANS_ID)
    for k in range(K + 1)
]

ks = list(range(K + 1))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ks, ll_metric_clean,   label="clean  (Japan)",  marker="o", markersize=3)
ax.plot(ks, ll_metric_corrupt, label="corrupt (France)", marker="s", markersize=3, linestyle="--")
ax.axhline(0.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("layer k  (0=embed, K=norm)")
ax.set_ylabel("logit( Tokyo) - logit( Paris)")
ax.set_title("Logit Lens — metric_k  (logit difference)")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(outputs_dir / "nb02_logit_lens_metric.png", dpi=150)
plt.show()
print("Saved: outputs/nb02_logit_lens_metric.png")


## 9. Activation Patching の概念と特定 layer のパッチ

### 概念

Activation Patching（残差ストリームパッチ）の手順：

1. **clean run** を実行 → 各 layer の hidden state `clean_hs[k]` を記録
2. **corrupt run** に hook を設置：layer k の出力を `clean_hs[k][0, clean_pos, :]` で上書き
3. corrupt run を再実行 → 最終 logits を記録
4. **回復率 (recovery)** を計算：

$$\text{recovery}(k) = \frac{\text{metric}(\text{patched}) - \text{metric}(\text{corrupt})}{\text{metric}(\text{clean}) - \text{metric}(\text{corrupt})}$$

metric = `logit( Tokyo) - logit( Paris)`  at last position

- recovery ≈ 0 → layer k のパッチは効果なし
- recovery ≈ 1 → layer k のパッチで clean run に完全回復

### hidden_states インデックスと patch site の対応

| k | patch site | モジュール |
|---|---|---|
| 0 | embed_tokens 出力 | `model.model.embed_tokens` |
| 1 ≤ k ≤ K-1 | layers[k-1] 出力 | `model.model.layers[k-1]` |
| K | norm 出力 | `model.model.norm` |


In [ ]:
def run_patch(k: int) -> torch.Tensor:
    """
    corrupt run の位置 corrupt_pos における layer k の残差ストリームを
    clean_hs[k][0, clean_pos, :] で置換して最終 logits [vocab] を返す。

    patch_vec は clean run 側の clean_pos から取る。
    今回は clean_pos == corrupt_pos だが、意味としては clean run 側の位置を使う。

    Transformers 5.x: Qwen3DecoderLayer.forward() は tensor を直接返す（tuple でない）。
    hook は out.clone() してから置換し、tensor を return する。
    """
    patch_vec = clean_hs[k][0, clean_pos, :].to(device)

    if k == 0:
        target_module = model.model.embed_tokens
    elif k < K:
        target_module = model.model.layers[k - 1]
    else:
        target_module = model.model.norm

    def hook(module, inp, out):
        out = out.clone()
        out[0, corrupt_pos, :] = patch_vec
        return out

    handle = target_module.register_forward_hook(hook)
    try:
        with torch.no_grad():
            patched_out = model(
                **corrupt_inputs,
                output_hidden_states=False,
                output_attentions=False,
                use_cache=False,
            )
    finally:
        handle.remove()

    return patched_out.logits[0, corrupt_pos, :].float()


In [ ]:
# metric のベースライン
clean_metric   = metric(clean_logits,   CLEAN_ANS_ID, CORRUPT_ANS_ID)
corrupt_metric = metric(corrupt_logits, CLEAN_ANS_ID, CORRUPT_ANS_ID)
print(f"clean   metric = {clean_metric:.4f}   logit(Tokyo) - logit(Paris)")
print(f"corrupt metric = {corrupt_metric:.4f}   logit(Tokyo) - logit(Paris)")
print(f"difference     = {clean_metric - corrupt_metric:.4f}")


In [ ]:
# k=24 のパッチ（効果が小さいはず）
print("=== k=24 patch ===")
plogits_24 = run_patch(24)
m24  = metric(plogits_24, CLEAN_ANS_ID, CORRUPT_ANS_ID)
rec24 = (m24 - corrupt_metric) / (clean_metric - corrupt_metric)
print(f"patched metric = {m24:.4f}")
print(f"recovery       = {rec24:.4f}")
display(topk_table(plogits_24, tokenizer, k=5))


In [ ]:
# k=25 のパッチ（回復が始まるはず）
print("=== k=25 patch ===")
plogits_25 = run_patch(25)
m25  = metric(plogits_25, CLEAN_ANS_ID, CORRUPT_ANS_ID)
rec25 = (m25 - corrupt_metric) / (clean_metric - corrupt_metric)
print(f"patched metric = {m25:.4f}")
print(f"recovery       = {rec25:.4f}")
display(topk_table(plogits_25, tokenizer, k=5))


### 注意：Logit Lens と Activation Patching は異なる操作

上の k=25 の結果を見ると、k=25 のパッチで recovery が大きく上がっている。
しかし、Section 7–8 の Logit Lens では、clean run の k=25 時点ではまだ ` Tokyo` の確率は高くない。

これは矛盾ではなく、**両手法が異なる操作**であることを示している。

| 操作 | 内容 |
|------|------|
| **Logit Lens** | 層 k の残差ストリームを**その場で** lm_head（+norm）に通して読む |
| **Activation Patching** | 層 k の残差ストリームを差し替えたあと、**残りの層を通常通り forward** させる |

したがって、「層 k の logit lens で ` Tokyo` がまだ強く見えない」ことは、
「その層の表現が最終出力に影響しない」ことを**意味しない**。

k=25 の残差ストリームは、lm_head で直接読んだ時点ではまだ ` Tokyo` を強く示さないが、
残りの 11 層（k=26〜K）がその表現をさらに処理することで、最終的に ` Tokyo` が強く出る状態に変換される。


## 10. 全 layer スイープ

In [ ]:
RUN_FULL_PATCHING = True   # False にすると outputs/ の既存 CSV から読む

if RUN_FULL_PATCHING:
    sweep_rows = []
    print(f"Patching sweep: k = 0 ... {K}")
    for k in range(K + 1):
        plogits = run_patch(k)
        pm  = metric(plogits, CLEAN_ANS_ID, CORRUPT_ANS_ID)
        rec = (pm - corrupt_metric) / (clean_metric - corrupt_metric)
        pprobs = torch.softmax(plogits, dim=-1)
        site = "embed" if k == 0 else (f"L{k-1:02d}" if k < K else "norm")
        sweep_rows.append({
            "k": k,
            "site": site,
            "recovery": rec,
            "patched_metric": pm,
            "p_tokyo_patched": pprobs[CLEAN_ANS_ID].item(),
            "p_paris_patched": pprobs[CORRUPT_ANS_ID].item(),
        })
        if k % 5 == 0 or k == K:
            print(f"  k={k:2d} ({site:6s}): recovery={rec:.4f}")

    df_sweep = pd.DataFrame(sweep_rows)
    df_sweep.to_csv(outputs_dir / "nb02_patching_sweep.csv", index=False)
    print(f"\nSaved: outputs/nb02_patching_sweep.csv")

else:
    csv_path = outputs_dir / "nb02_patching_sweep.csv"
    if csv_path.exists():
        df_sweep = pd.read_csv(csv_path)
        print(f"Loaded from {csv_path}")
    else:
        # script 12 の出力を流用
        csv_path2 = outputs_dir / "prelim_residual_patching_by_layer.csv"
        df_tmp = pd.read_csv(csv_path2)
        df_sweep = pd.DataFrame(df_tmp[["layer_k", "patch_site", "recovery"]]).rename(
            columns={"layer_k": "k", "patch_site": "site"}
        )
        print(f"Loaded from {csv_path2}  (script 12 output)")

display(df_sweep.head(10))


## 11. Recovery curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_sweep["k"], df_sweep["recovery"], marker="o", markersize=4, color="steelblue", label="recovery")
ax.axhline(0.0, color="gray", linestyle="--", linewidth=0.8)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("patch site k  (0=embed, K=norm)")
ax.set_ylabel("recovery")
ax.set_title("Activation Patching — Recovery by Layer\n"
             "(clean→corrupt injection at last token position)")
ax.set_xticks(range(0, K + 1, 4))
ax.set_ylim(-0.1, 1.1)
ax.grid(True, alpha=0.3)

# 最初に recovery >= 0.5 になる k にマーカー
candidates = df_sweep.loc[df_sweep["recovery"] >= 0.5, "k"]
if len(candidates) > 0:
    k_transition = int(candidates.min())
    ax.axvline(k_transition, color="tomato", linestyle=":", linewidth=1.5,
               label=f"k={k_transition}: recovery≥0.5")
    ax.legend()

plt.tight_layout()
plt.savefig(outputs_dir / "nb02_recovery_curve.png", dpi=150)
plt.show()
print("Saved: outputs/nb02_recovery_curve.png")
if len(candidates) > 0:
    print(f"最初に recovery≥0.5 になる layer: k={k_transition}")


## 12. 特定 layer のパッチ後 top-k 比較

In [ ]:
# k=K のパッチ後 logits
plogits_K = run_patch(K)

labels = [
    ("clean baseline",   clean_logits),
    ("corrupt baseline", corrupt_logits),
    ("patch k=24",       plogits_24),
    ("patch k=25",       plogits_25),
    (f"patch k=K={K}",   plogits_K),
]

for label, lgts in labels:
    rec_val = (metric(lgts, CLEAN_ANS_ID, CORRUPT_ANS_ID) - corrupt_metric) / (clean_metric - corrupt_metric)
    print(f"=== {label}  (recovery={rec_val:.4f}) ===")
    display(topk_table(lgts, tokenizer, k=5))
    print()


## 13. k=K サニティチェック

In [ ]:
# k=K (norm 後) のパッチは clean run の最終隠れ状態をそのまま注入するため recovery=1.000 になるはず
plogits_K_check = run_patch(K)
mK   = metric(plogits_K_check, CLEAN_ANS_ID, CORRUPT_ANS_ID)
recK = (mK - corrupt_metric) / (clean_metric - corrupt_metric)
print(f"k=K patch:")
print(f"  patched metric = {mK:.6f}")
print(f"  clean   metric = {clean_metric:.6f}")
print(f"  recovery       = {recK:.6f}")
print()
if abs(recK - 1.0) < 1e-3:
    print("OK: k=K で recovery=1.000 を確認 (hook の実装が正しい)")
else:
    print(f"[warning] k=K recovery={recK:.6f} != 1.000  (hook の実装を確認)")


## 14. まとめ

### 観察結果

| 手法 | 観察したこと |
|------|-------------|
| **Logit Lens** | 前半層（k≤24程度）では top-1 が意味のないトークン、後半層（k≈25以降）から正答 ` Tokyo` が上位に現れる |
| **Activation Patching** | k≤24 では recovery≈0、k=25 付近から急増、k≥34 では recovery≈1.0 |
| **k=K sanity check** | k=K（norm 後）のパッチで recovery=1.000 → hook の実装が正しいことを確認 |

### 解釈（この prompt pair とこの setup における観察）

- この題材・この last-token position・この metric では、**k=25 付近の残差ストリームを差し替えると最終出力が ` Tokyo` 側へ大きく変化した**。
- 少なくとも last-token position の残差ストリームを単独で差し替えるこの実験では、k≤24 の patch は最終出力をほとんど変えなかった。
- ただし、これを「知識が一般に後半層だけにある」と一般化してはいけない。あくまで **この prompt pair・この patching setup における観察**である。
- Logit Lens では ` Tokyo` の確率が明確に上がるのは k≈29 以降だが、Activation Patching では k=25 でも大きな recovery が得られた。これは両手法が**異なる操作**であることを反映している（詳細は Section 9 の注記を参照）。

### 参考文献

- Logit Lens: Nostalgebraist (2020), *Interpreting GPT: the logit lens*
- Activation Patching / Causal Tracing: Meng et al. (2022), ROME
- TransformerLens: Nanda et al. (2022)
